# Gaia BH3: a Gaia DR4 EAS astrometry only fit

Fit source **4318465066420528000** from the local EAS prerelease XML.
The luminous star is body 0 and the dark black hole companion is body 1.
Both masses are in **solar masses**.

Data credit: **ESA/Gaia/DPAC**. See the
[data provenance](../../orbitize/example_data/gaia_dr4_eas_README.md),
[ESA's DR4 BH3 example](https://github.com/esa/gaia-bhthree/tree/gaia-dr4-prerelease),
and [Panuzzo et al. (2024)](https://arxiv.org/abs/2404.10486).
This is a first exploratory fit, not a claim of convergence.

In [ ]:
from pathlib import Path
from copy import copy
from time import perf_counter
import sys
from astropy.time import Time

repo_root = Path("../..").resolve()
if not (repo_root / "orbitize").is_dir():
    raise FileNotFoundError("Run this notebook from orbitize/docs/tutorials.")
sys.path.insert(0, str(repo_root))

import corner
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
import orbitize
from orbitize import gaia, plot, priors, read_input, sampler, system

# Seed both NumPy interfaces once here: sampler and bootstrap, respectively.
np.random.seed(42)
rng = np.random.default_rng(42)
plt.style.use("default")

source_id = 4318465066420528000
ref_epoch_mjd = 57936.375  # J2017.5, MJD(TCB)
stellar_mass = 0.76  # Msun; Panuzzo et al. (2024), Table 3
stellar_mass_err = 0.05  # Msun

## 1. Read and combine the CCD measurements

Keep the AGIS-used CCDs and combine them once per transit using the existing
bootstrap helper (500 resamples, with the IPD error floor). These uncertainties
stay fixed during MCMC. A separate sampled jitter is added in quadrature.



In [ ]:
xml_path = Path(orbitize.DATADIR) / "GAIA_DR4_PRERELEASE_EPOCH_ASTROMETRY_RAW.xml"
ccd_data = read_input.from_dl2(xml_path, source_id)
data = read_input.combine_transits(ccd_data, n_boot=500, rng=rng)

time_mjd = np.asarray(data["obs_time_tcb"], dtype=float)
al = np.asarray(data["centroid_pos_al"], dtype=float)
al_err = np.asarray(data["centroid_pos_error_al"], dtype=float)
psi = np.asarray(data["scan_pos_angle"], dtype=float)
parallax_factor = np.asarray(data["parallax_factor_al"], dtype=float)
dt_yr = (time_mjd - ref_epoch_mjd) / 365.25
sinpsi, cospsi = np.sin(psi), np.cos(psi)

assert np.all(
    np.isfinite(np.column_stack((time_mjd, al, al_err, psi, parallax_factor)))
)
assert np.all(al_err > 0)
print(f"{len(ccd_data)} CCD measurements -> {len(data)} transits")
print(f"Observed baseline: {np.ptp(time_mjd) / 365.25:.2f} years")
data[:5]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(time_mjd, al, yerr=al_err, fmt="k.", alpha=0.8)
ax.set(
    xlabel="Epoch [MJD(TCB)]",
    ylabel="Along-scan position [mas]",
    title="Gaia BH3: reduced EAS data",
)
fig.tight_layout()

## 2. System and priors

Let's use the **Period** basis. Period is log-uniform over 1–50 years, black hole mass log-uniform over
1–100 Msun, eccentricity uniform over 0–0.95, inclination isotropic,
and both angles and periastron phase uniform.

The luminous star mass has a positive Gaussian prior, 0.76 +/- 0.05 Msun
([Table 3](https://arxiv.org/html/2404.10486v2#S2.T3)). That stellar estimate
used a Gaia-based distance. **Parallax is fitted with a uniform 0.1–5 mas
prior**. Jitter is log-uniform over 0.0001–100 mas. The four linear terms retain the
current basis defaults: +/-10,000 mas for offsets and +/-10,000 mas/yr for PM.

The current System API still requires a typed data table. The empty table
below contains **zero observations**; all likelihood data enter through
`gaia=dr4`. No fake astrometry is added, and the library files are unchanged.

In [ ]:
dr4 = gaia.DR4LogProb(
    source_id,
    data,
    ref_epoch_mjd=ref_epoch_mjd,
    jitter_prior=priors.LogUniformPrior(1e-4, 100),
)

my_sys = system.System(
    1,
    None,
    stellar_mass,
    2.0,
    mass_err=stellar_mass_err,
    plx_err=1.0,
    fit_secondary_mass=True,
    gaia=dr4,
    tau_ref_epoch=ref_epoch_mjd,
    fitting_basis="Period",
)
idx = my_sys.param_idx

# Replace the constructor's placeholder parallax prior before sampling.
for name, prior in {
    "per1": priors.LogUniformPrior(1.0, 50.0),
    "ecc1": priors.UniformPrior(0.0, 0.95),
    "inc1": priors.SinPrior(),
    "aop1": priors.UniformPrior(0.0, 2 * np.pi),
    "pan1": priors.UniformPrior(0.0, 2 * np.pi),
    "tau1": priors.UniformPrior(0.0, 1.0),
    "m1": priors.LogUniformPrior(1.0, 100.0),
    "plx": priors.UniformPrior(0.1, 5.0),
}.items():
    my_sys.sys_priors[idx[name]] = prior

assert len(my_sys.data_table) == 0
assert len(my_sys.rv_instruments) == 0
assert dr4.n_obs == len(data)
print("Basis:", my_sys.fitting_basis, "| Gaia transits:", dr4.n_obs, "| RVs: 0")

## 3. Initialize broad walkers

Draw every orbital parameter, parallax, mass and jitter from its prior.
For each starting orbit, subtract its orbital and parallax contribution and
solve only for the four linear position/PM terms:

$$y_i = \eta_i - \varpi P_i - \eta_{\mathrm{orb},i},$$
$$y_i \approx (\alpha_0 + \mu_{\alpha*}\Delta t_i)\sin\psi_i
             + (\delta_0 + \mu_\delta\Delta t_i)\cos\psi_i.$$

The linear terms are freely sampled once MCMC starts.

In [ ]:
num_temps = 10
num_walkers = 128
num_threads = 10
burn_steps = 2_000
num_steps = 5_000
thin = 1

mcmc = sampler.MCMC(
    my_sys,
    num_temps=num_temps,
    num_walkers=num_walkers,
    num_threads=num_threads,
)
sampled_idx = mcmc.sampled_param_idx
positions = mcmc.curr_pos.reshape(-1, mcmc.num_params)
full_positions = mcmc._fill_in_fixed_params(positions)
standard_positions = my_sys.basis.to_standard_basis(full_positions.T.copy())
ra, dec, _, _ = my_sys.compute_all_orbits(standard_positions, epochs=dr4.epochs_mjd)
orbit_al = ra[:, 0, :] * sinpsi[:, None] + dec[:, 0, :] * cospsi[:, None]

linear_columns = [sampled_idx[name] for name in ("alpha0", "delta0", "pm_ra", "pm_dec")]

linear_epochs = Time(dr4.epochs_mjd, format="mjd", scale="tcb")
linear_ref = Time(ref_epoch_mjd, format="mjd", scale="tcb").jyear

for walker, params in enumerate(full_positions):
    sigma = np.hypot(al_err, params[idx["dr4_jitter"]])
    target = al - params[idx["plx"]] * parallax_factor - orbit_al[:, walker]
    positions[walker, linear_columns] = gaia.HGCALogProb._linear_pm_fit(
        epochs=linear_epochs,
        raoff_planet=target * sinpsi,
        decoff_planet=target * cospsi,
        epoch_ref_ra=linear_ref,
        epoch_ref_dec=linear_ref,
        sin_phi=cospsi,
        cos_phi=sinpsi,
        errs=sigma,
    )
mcmc.check_prior_support()
print("Initialized broad walkers")

## 4. Run once, then inspect

The next cell starts the fit.
These settings are a computational budget to inspect orientation, period, mass and jitter.


In [ ]:
start = perf_counter()
run = mcmc.run_sampler(num_walkers * num_steps, burn_steps=burn_steps, thin=thin)
post = mcmc.results.post
print(f"Run took {(perf_counter() - start) / 60:.1f} minutes")
print("Mean acceptance per temperature:", np.mean(run.acceptance_fraction, axis=1))
print("Temperature-swap acceptance:", run.tswap_acceptance_fraction)
print(f"Best log-likelihood: {np.nanmax(mcmc.results.lnlike):.3f}")

In [ ]:
chain_params = ["per1", "ecc1", "inc1", "pan1", "m1", "plx", "dr4_jitter"]
chain_labels = [
    "P [yr]",
    "e",
    "i [rad]",
    "LAN [rad]",
    "BH mass [Msun]",
    "Parallax [mas]",
    "Gaia jitter [mas]",
]
chain_figures = mcmc.examine_chains(
    param_list=chain_params,
    n_walkers=16,
    transparency=0.35,
)
for fig, label in zip(chain_figures, chain_labels):
    fig.axes[0].set_ylabel(label)
    fig.tight_layout()

## 5. Corner Plot

Let's take a look at the corner plot for BH3. Orange lines show true parameters from the best fit in the paper.

In [ ]:
corner_params = ["per1", "ecc1", "inc1", "m1", "m0", "plx", "dr4_jitter"]
draws = rng.choice(len(post), size=min(20_000, len(post)), replace=False)
corner_samples = post[draws][:, [idx[name] for name in corner_params]].copy()
corner_samples[:, 0] *= 365.25
corner_samples[:, 2] = np.rad2deg(corner_samples[:, 2])

corner_figure = corner.corner(
    corner_samples,
    labels=[
        "P [day]",
        "e",
        "i [deg]",
        "BH mass [Msun]",
        "Star mass [Msun]",
        "Parallax [mas]",
        "Gaia jitter [mas]",
    ],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    plot_datapoints=False,
    fill_contours=True,
    max_n_ticks=3,
)
# Panuzzo et al. (2024), combined solution: Tables 2 and 3 and Sect. 3.
published_fit = [4253.1, 0.7291, 110.580, 32.70, 0.76, 1.6933, None]
corner.overplot_lines(corner_figure, published_fit, color="tab:orange", linestyle="--")
corner_figure.axes[0].plot(
    [], [], color="tab:orange", linestyle="--", label="Panuzzo et al. (2024), Gaia+RV"
)
corner_figure.legend(
    *corner_figure.axes[0].get_legend_handles_labels(), loc="upper right", frameon=False
)

In [ ]:
best_idx = np.nanargmax(mcmc.results.lnlike)
best = post[best_idx].copy()
best_results = copy(mcmc.results)
best_results.post = post[[best_idx]]
best_results.lnlike = np.asarray(mcmc.results.lnlike)[[best_idx]]

orbit_figure = best_results.plot_dr4_orbit(
    my_sys,
    include_planet=True,
    num_orbits_to_plot=1,
    num_epochs_to_plot=500,
    data_params=best,
    include_jitter=True,
    fontsize=14,
)
orbit_figure.axes[0].set_title("Gaia BH3: best-likelihood stellar orbit (Gaia only)")

## 6. Save the run

This writes the posterior, likelihoods, system and Gaia observations to HDF5
in the current notebook directory. Use a new filename for each comparison run.

In [ ]:
from pathlib import Path
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_path = Path(f"Gaia_DR4_BH3_results_{stamp}.hdf5")
mcmc.results.save_results(str(output_path))
print("Saved:", output_path.resolve())